# Overview
Output : grapheme_root, vowel_diacritics, consonant_diacritics, and grapheme_id
#### Versions
- SEResNeXt001<br>
Baseline Model Built. <br>
- SEResNeXt002<br>
Cutmix applied. No center crop, Just resize. Minimum Augmentation. SGD to Adam. LR min to 8e-6<br>
32 Split 0 Fold : 0.9843<br>
- SEResNeXt003<br>
Center crop.<br>
32 Split 0 Fold : 0.9817<br>
- SEResNeXt004<br>
No center center crop. Alpha=1.0, Mixup Added. <br>
32 Split 0 Fold : 0.9849<br>
- SEResNeXt005<br>
Batch Size 512. <br>
32 Split 0 Fold : 0.9812<br>
- SEResNeXt006<br>
Batch Size 256. Add Elastic Distortion<br>
32 Split 0 Fold : 0.9845<br>
- SEResNeXt007<br>
RAdam, Grid Mask, Elastic Distortion 0.2, eta_min=4e-6, epoch 120<br>
32 Split 0 Fold : 0.9891<br>
- SEResNeXt008<br>
FMix 0.25, Cutmix 0.25, Mixup 0.25, image size 192<br>
32 Split 0 Fold : 0.9901<br>
- SEResNeXt009<br>
Cutmix 0.33, Mixup 0.33, No FMix, image size 192, Noticed an error on Cutmix/Mixup -> changed to apply per batch, not per epoch. <br>
5 Split 0 Fold : 0.9894<br>
- SEResNeXt010<br>
No Cutmix, Mixup 1.00, No FMix, GridMask 1.00, image size 192. <br>
5 Split 0 Fold : 0.9860 with 60 epochs. <br>
- SEResNeXt011<br>
Cutmix 0.5, Mixup 0.5, No FMix, GridMask 1.00, image size 64 for tuning. <br>
5 Split 0 Fold : 0.9733<br>
- SEResNeXt012<br>
Added OHEM loss. <br>
5 Split 0 Fold : 0.9699<br>
- SEResNeXt101<br>
L2 Softmax on Grapheme ID. <br>
5 Split 0 Fold : 0.9860<br>
- SEResNeXt013<br>
Version 008 with 5 Split 0 Fold. Transfer weights from 009. <br>
5 Split 0 Fold : 0.9899<br>
- SEResNeXt014<br>
Version 009 with GeM. Transfer weights from 009. image size 224. <br>
5 Split 0 Fold : 0.9908<br>

In [1]:
import numpy as np
import pandas as pd
import os, pickle, gc, random, tqdm, cv2, six, math, datetime

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import recall_score
from skimage.transform import AffineTransform, warp
import albumentations as A
from collections import OrderedDict
import scipy.stats as scs

In [2]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.nn.parameter import Parameter
from torch.utils.data.dataloader import DataLoader
from torch.utils.data.dataset import Dataset

!pip install ../module/pretrainedmodels-0.7.4/ > /dev/null
import pretrainedmodels

In [3]:
VERSION = 'SEResNeXt014'
TARGET_COLS = ['grapheme_root', 'vowel_diacritic', 'consonant_diacritic', 'grapheme_id']
TRAINING = True
VALIDATION = True
PSEUDO_LABELLING = False
LOCAL_PATH = '../input'
WEIGHT_PATH = '../input/weights'
PREPROCESSED_DATA_PATH = '../input/parquet-to-feather-no-'
IMAGENET_MODEL_WEIGHT_FILE = '../input/weights/se_resnext50_32x4d-a260b3a4.pth'
HEIGHT = 137
WIDTH = 236
IMAGE_SIZE = 224
N_SPLITS = 5
FOLD = [2]
SEED = 9253
BATCH_SIZE = 64
VALIDATION_BATCH_SIZE = 128
EPOCHS = 200
EPOCH_RELEASE = 1
EARLY_STOPPING = 50
BATCH_ACCUMULATION_COUNT = 4
DEVICE = 'cuda:1' if torch.cuda.is_available() else 'cpu'
NUM_WORKERS = 4

def seed_everything(seed: int):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    
seed_everything(SEED)

# 1. Load Datasets and Perform some EDA

In [4]:
train = pd.read_csv(LOCAL_PATH+'/train.csv')
NUM_TRAIN = len(train)
grapheme_unique_dataset = train[
    ['grapheme_root','vowel_diacritic','consonant_diacritic','grapheme']
].drop_duplicates().reset_index(drop=True)
grapheme_unique = grapheme_unique_dataset['grapheme'].values
train['grapheme_id'] = train['grapheme'].apply(lambda x: np.where(grapheme_unique==x)[0][0])
print('Number of unique graphemes : ', len(grapheme_unique))
print(
    'Number of unique patterns for grapheme_root, vowel_diacritic, and consonant_diacritic : ', 
    train[['grapheme_root','vowel_diacritic','consonant_diacritic']].drop_duplicates().shape[0]
)
train.head(2)

Number of unique graphemes :  1295
Number of unique patterns for grapheme_root, vowel_diacritic, and consonant_diacritic :  1292


,image_id,grapheme_root,vowel_diacritic,consonant_diacritic,grapheme,grapheme_id
0,Train_0,15,9,5,ক্ট্রো,0
1,Train_1,159,0,0,হ,1


In [5]:
test = pd.read_csv(LOCAL_PATH+'/test.csv')
NUM_TEST = int(len(test) / 3)
test.head(2)

,row_id,image_id,component
0,Test_0_consonant_diacritic,Test_0,consonant_diacritic
1,Test_0_grapheme_root,Test_0,grapheme_root


In [6]:
grapheme_dict = grapheme_unique_dataset.to_dict('index')

print('Number of graphemes in grapheme_dict : ', len(grapheme_dict))
print('3 Header Samples of grapheme_dict : ')
dict(list(grapheme_dict.items())[0:3])

Number of graphemes in grapheme_dict :  1295
3 Header Samples of grapheme_dict : 


{0: {'grapheme_root': 15,
  'vowel_diacritic': 9,
  'consonant_diacritic': 5,
  'grapheme': 'ক্ট্রো'},
 1: {'grapheme_root': 159,
  'vowel_diacritic': 0,
  'consonant_diacritic': 0,
  'grapheme': 'হ'},
 2: {'grapheme_root': 22,
  'vowel_diacritic': 3,
  'consonant_diacritic': 5,
  'grapheme': 'খ্রী'}}

# 2. Preprocessing : Define Dataset Generator

In [7]:
################
# Crop and Resize
# https://www.kaggle.com/iafoss/image-preprocessing-128x128
################

def bbox(img):
    rows = np.any(img, axis=1)
    cols = np.any(img, axis=0)
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    return rmin, rmax, cmin, cmax

def crop_resize(img0, size=IMAGE_SIZE, pad=16):
    #crop a box around pixels large than the threshold 
    #some images contain line at the sides
    ymin,ymax,xmin,xmax = bbox(img0[5:-5,5:-5] > 80)
    #cropping may cut too much, so we need to add it back
    xmin = xmin - 13 if (xmin > 13) else 0
    ymin = ymin - 10 if (ymin > 10) else 0
    xmax = xmax + 13 if (xmax < WIDTH - 13) else WIDTH
    ymax = ymax + 10 if (ymax < HEIGHT - 10) else HEIGHT
    img = img0[ymin:ymax,xmin:xmax]
    #remove lo intensity pixels as noise
    img[img < 28] = 0
    lx, ly = xmax-xmin,ymax-ymin
    l = max(lx,ly) + pad
    #make sure that the aspect ratio is kept in rescaling
    img = np.pad(img, [((l-ly)//2,), ((l-lx)//2,)], mode='constant')
    return cv2.resize(img,(size,size))

def plain_resize(img, size=IMAGE_SIZE):
    return cv2.resize(img,(size,size))


In [8]:
################
# Augmentations
################

def affine_image(img):
    """
    Args:
        img: (h, w) or (1, h, w)
    Returns:
        img: (h, w)
    """
    if img.ndim == 3:
        img = img[0]
        
    # --- scale ---
    min_scale = 0.8
    max_scale = 1.2
    sx = np.random.uniform(min_scale, max_scale)
    sy = np.random.uniform(min_scale, max_scale)

    # --- rotation ---
    max_rot_angle = 7
    rot_angle = np.random.uniform(-max_rot_angle, max_rot_angle) * np.pi / 180.

    # --- shear ---
    max_shear_angle = 10
    shear_angle = np.random.uniform(-max_shear_angle, max_shear_angle) * np.pi / 180.

    # --- translation ---
    max_translation = 4
    tx = np.random.randint(-max_translation, max_translation)
    ty = np.random.randint(-max_translation, max_translation)

    tform = AffineTransform(scale=(sx, sy), rotation=rot_angle, shear=shear_angle,
                            translation=(tx, ty))
    transformed_image = warp(img, tform)
    assert transformed_image.ndim == 2
    return transformed_image

class GridMask(A.core.transforms_interface.DualTransform):
    """GridMask augmentation for image classification and object detection.
    
    Author: Qishen Ha
    Email: haqishen@gmail.com
    2020/01/29

    Args:
        num_grid (int): number of grid in a row or column.
        fill_value (int, float, lisf of int, list of float): value for dropped pixels.
        rotate ((int, int) or int): range from which a random angle is picked. If rotate is a single int
            an angle is picked from (-rotate, rotate). Default: (-90, 90)
        mode (int):
            0 - cropout a quarter of the square of each grid (left top)
            1 - reserve a quarter of the square of each grid (left top)
            2 - cropout 2 quarter of the square of each grid (left top & right bottom)

    Targets:
        image, mask

    Image types:
        uint8, float32

    Reference:
    |  https://arxiv.org/abs/2001.04086
    |  https://github.com/akuxcw/GridMask
    """

    def __init__(self, num_grid=3, fill_value=0, rotate=0, mode=0, always_apply=False, p=0.5):
        super(GridMask, self).__init__(always_apply, p)
        if isinstance(num_grid, int):
            num_grid = (num_grid, num_grid)
        if isinstance(rotate, int):
            rotate = (-rotate, rotate)
        self.num_grid = num_grid
        self.fill_value = fill_value
        self.rotate = rotate
        self.mode = mode
        self.masks = None
        self.rand_h_max = []
        self.rand_w_max = []

    def init_masks(self, height, width):
        if self.masks is None:
            self.masks = []
            n_masks = self.num_grid[1] - self.num_grid[0] + 1
            for n, n_g in enumerate(range(self.num_grid[0], self.num_grid[1] + 1, 1)):
                grid_h = height / n_g
                grid_w = width / n_g
                this_mask = np.ones((int((n_g + 1) * grid_h), int((n_g + 1) * grid_w))).astype(np.uint8)
                for i in range(n_g + 1):
                    for j in range(n_g + 1):
                        this_mask[
                             int(i * grid_h) : int(i * grid_h + grid_h / 2),
                             int(j * grid_w) : int(j * grid_w + grid_w / 2)
                        ] = self.fill_value
                        if self.mode == 2:
                            this_mask[
                                 int(i * grid_h + grid_h / 2) : int(i * grid_h + grid_h),
                                 int(j * grid_w + grid_w / 2) : int(j * grid_w + grid_w)
                            ] = self.fill_value
                
                if self.mode == 1:
                    this_mask = 1 - this_mask

                self.masks.append(this_mask)
                self.rand_h_max.append(grid_h)
                self.rand_w_max.append(grid_w)

    def apply(self, image, mask, rand_h, rand_w, angle, **params):
        h, w = image.shape[:2]
        mask = A.augmentations.functional.rotate(mask, angle) if self.rotate[1] > 0 else mask
        mask = mask[:,:,np.newaxis] if image.ndim == 3 else mask
        image *= mask[rand_h:rand_h+h, rand_w:rand_w+w].astype(image.dtype)
        return image

    def get_params_dependent_on_targets(self, params):
        img = params['image']
        height, width = img.shape[:2]
        self.init_masks(height, width)

        mid = np.random.randint(len(self.masks))
        mask = self.masks[mid]
        rand_h = np.random.randint(self.rand_h_max[mid])
        rand_w = np.random.randint(self.rand_w_max[mid])
        angle = np.random.randint(self.rotate[0], self.rotate[1]) if self.rotate[1] > 0 else 0

        return {'mask': mask, 'rand_h': rand_h, 'rand_w': rand_w, 'angle': angle}

    @property
    def targets_as_params(self):
        return ['image']

    def get_transform_init_args_names(self):
        return ('num_grid', 'fill_value', 'rotate', 'mode')

def add_gaussian_noise(x, sigma):
    x += np.random.randn(*x.shape) * sigma
    x = np.clip(x, 0., 1.)
    return x

def _evaluate_ratio(ratio):
    if ratio <= 0.:
        return False
    return np.random.uniform() < ratio

def apply_aug(aug, image):
    return aug(image=image)['image']

class Transform:
    def __init__(self, affine=True, crop=True, 
                 normalize=True, train=True, 
                 sigma=-1., blur_ratio=0., noise_ratio=0., cutout_ratio=0.,
                 grid_distortion_ratio=0., elastic_distortion_ratio=0., random_brightness_ratio=0.,
                 piece_affine_ratio=0., ssr_ratio=0., grid_mask_ratio=0.):
        self.affine = affine
        self.crop = crop
        self.normalize = normalize
        self.train = train
        self.sigma = sigma / 255.

        self.blur_ratio = blur_ratio
        self.noise_ratio = noise_ratio
        self.cutout_ratio = cutout_ratio
        self.grid_distortion_ratio = grid_distortion_ratio
        self.elastic_distortion_ratio = elastic_distortion_ratio
        self.random_brightness_ratio = random_brightness_ratio
        self.piece_affine_ratio = piece_affine_ratio
        self.ssr_ratio = ssr_ratio
        self.grid_mask_ratio = grid_mask_ratio

    def __call__(self, example):
        if self.train:
            x, y = example
        else:
            x = example
            
        # --- Augmentation ---
        if self.affine:
            x = affine_image(x)

        # --- Gaussian Noise ---
        if self.sigma > 0.:
            x = add_gaussian_noise(x, sigma=self.sigma)

        # albumentations...
        x = x.astype(np.float32)
        assert x.ndim == 2
        # 1. blur
        if _evaluate_ratio(self.blur_ratio):
            r = np.random.uniform()
            if r < 0.25:
                x = apply_aug(A.Blur(p=1.0), x)
            elif r < 0.5:
                x = apply_aug(A.MedianBlur(blur_limit=5, p=1.0), x)
            elif r < 0.75:
                x = apply_aug(A.GaussianBlur(p=1.0), x)
            else:
                x = apply_aug(A.MotionBlur(p=1.0), x)

        if _evaluate_ratio(self.noise_ratio):
            r = np.random.uniform()
            if r < 0.50:
                x = apply_aug(A.GaussNoise(var_limit=5. / 255., p=1.0), x)
            else:
                x = apply_aug(A.MultiplicativeNoise(p=1.0), x)

        if _evaluate_ratio(self.cutout_ratio):
            # A.Cutout(num_holes=2,  max_h_size=2, max_w_size=2, p=1.0)  # Deprecated...
            x = apply_aug(A.CoarseDropout(max_holes=8, max_height=8, max_width=8, p=1.0), x)

        if _evaluate_ratio(self.grid_distortion_ratio):
            x = apply_aug(A.GridDistortion(p=1.0), x)

        if _evaluate_ratio(self.elastic_distortion_ratio):
            x = apply_aug(A.ElasticTransform(
                sigma=50, alpha=1, alpha_affine=10, p=1.0), x)

        if _evaluate_ratio(self.random_brightness_ratio):
            # A.RandomBrightness(p=1.0)  # Deprecated...
            # A.RandomContrast(p=1.0)    # Deprecated...
            x = apply_aug(A.RandomBrightnessContrast(p=1.0), x)

        if _evaluate_ratio(self.piece_affine_ratio):
            x = apply_aug(A.IAAPiecewiseAffine(p=1.0), x)

        if _evaluate_ratio(self.ssr_ratio):
            x = apply_aug(A.ShiftScaleRotate(
                shift_limit=0.0625,
                scale_limit=0.1,
                rotate_limit=30,
                p=1.0), x)
            
        if _evaluate_ratio(self.grid_mask_ratio):
            x = apply_aug(A.OneOf([
                GridMask(num_grid=(2,5), rotate=15, mode=0, always_apply=True, p=1.0),
                GridMask(num_grid=(2,3), rotate=15, mode=2, always_apply=True, p=1.0),
            ], p=1.0), x)

        if self.normalize:
            x = (x.astype(np.float32) - 0.0692) / 0.2051
        if x.ndim == 2:
            x = x[None, :, :]
        x = x.astype(np.float32)
        if self.train:
            y = y.astype(np.int64)
            return x, y
        else:
            return x
        

In [9]:
################
# Dataset Generator
# https://www.kaggle.com/corochann/bengali-seresnext-training-with-pytorch/data
################

def prepare_images(data_type='train', indices=[0, 1, 2, 3]):
    assert data_type in ['train', 'test']
    if data_type=='test':
        images = np.zeros((NUM_TEST, IMAGE_SIZE, IMAGE_SIZE), dtype = "uint8")
    else:
        images = np.zeros((NUM_TRAIN, IMAGE_SIZE, IMAGE_SIZE), dtype = "uint8")
    sum_count = 0
    for idx in indices:
        if data_type=='test':
            df = pd.read_parquet(LOCAL_PATH+'/test_image_data_{}.parquet'.format(idx))
        else:
            df = pd.read_feather(PREPROCESSED_DATA_PATH+'{}/train_image_data_{}.feather'.format(idx,idx))
        all_images = 255 - df.iloc[:, 1:].values.reshape(-1, HEIGHT, WIDTH).astype(np.uint8)
        del df
        gc.collect()
        for i, image in tqdm.tqdm(enumerate(all_images)):
            image = (image*(255.0/image.max())).astype(np.uint8)
            image = plain_resize(image)
#             image = crop_resize(image)
            images[sum_count+i] = image
        sum_count += len(all_images)
        del all_images
        gc.collect()
    return images

class DatasetMixin(Dataset):
    def __init__(self, transform=None):
        self.transform = transform

    def __getitem__(self, index):
        """Returns an example or a sequence of examples."""
        if torch.is_tensor(index):
            index = index.tolist()
        if isinstance(index, slice):
            current, stop, step = index.indices(len(self))
            return [self.get_example_wrapper(i) for i in
                    six.moves.range(current, stop, step)]
        elif isinstance(index, list) or isinstance(index, np.ndarray):
            return [self.get_example_wrapper(i) for i in index]
        else:
            return self.get_example_wrapper(index)

    def __len__(self):
        """Returns the number of data points."""
        raise NotImplementedError

    def get_example_wrapper(self, i):
        """Wrapper of `get_example`, to apply `transform` if necessary"""
        example = self.get_example(i)
        if self.transform:
            example = self.transform(example)
        return example

    def get_example(self, i):
        """Returns the i-th example.
        Implementations should override it. It should raise :class:`IndexError`
        if the index is invalid.
        Args:
            i (int): The index of the example.
        Returns:
            The i-th example.
        """
        raise NotImplementedError
        
class BengaliAIDataset(DatasetMixin):
    def __init__(self, images, labels=None, transform=None, indices=None):
        super(BengaliAIDataset, self).__init__(transform=transform)
        self.images = images
        self.labels = labels
        if indices is None:
            indices = np.arange(len(images))
        self.indices = indices
        self.train = labels is not None

    def __len__(self):
        """return length of this dataset"""
        return len(self.indices)

    def get_example(self, i):
        """Return i-th data"""
        i = self.indices[i]
        x = self.images[i]
        # Opposite white and black: background will be white and
        # for future Affine transformation
        x = x.astype(np.float32) / 255.
        if self.train:
            y = self.labels[i]
            return x, y
        else:
            return x
        

# 3. Model Definition

In [10]:
"""
https://www.kaggle.com/satian/seresnext101-pytorch-starter
https://github.com/Cadene/pretrained-models.pytorch/blob/021d97897c9aa76ec759deff43d341c4fd45d7ba/pretrainedmodels/models/senet.py#L369
"""

pretrained_settings = {
    'se_resnext50_32x4d': {
        'imagenet': {
            'file_path': IMAGENET_MODEL_WEIGHT_FILE,
            'input_space': 'RGB',
            'input_size': [3, 224, 224],
            'input_range': [0, 1],
            'mean': [0.485, 0.456, 0.406],
            'std': [0.229, 0.224, 0.225],
            'num_classes': 1000
        }
    }
}


def gem(x, p=3, eps=1e-6):
    return F.avg_pool2d(x.clamp(min=eps).pow(p), (x.size(-2), x.size(-1))).pow(1./p)

class GeM(nn.Module):
    def __init__(self, p=3, eps=1e-6):
        super(GeM,self).__init__()
        self.p = Parameter(torch.ones(1)*p)
        self.eps = eps
    def forward(self, x):
        return gem(x, p=self.p, eps=self.eps)       
    def __repr__(self):
        return self.__class__.__name__ + '(' + 'p=' + '{:.4f}'.format(self.p.data.tolist()[0]) + ', ' + 'eps=' + str(self.eps) + ')'


class SEModule(nn.Module):

    def __init__(self, channels, reduction):
        super(SEModule, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Conv2d(channels, channels // reduction, kernel_size=1,
                             padding=0)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Conv2d(channels // reduction, channels, kernel_size=1,
                             padding=0)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        module_input = x
        x = self.avg_pool(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.sigmoid(x)
        return module_input * x


class Bottleneck(nn.Module):
    """
    Base class for bottlenecks that implements `forward()` method.
    """
    def forward(self, x):
        residual = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        if self.downsample is not None:
            residual = self.downsample(x)

        out = self.se_module(out) + residual
        out = self.relu(out)

        return out


class SEBottleneck(Bottleneck):
    """
    Bottleneck for SENet154.
    """
    expansion = 4

    def __init__(self, inplanes, planes, groups, reduction, stride=1,
                 downsample=None):
        super(SEBottleneck, self).__init__()
        self.conv1 = nn.Conv2d(inplanes, planes * 2, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes * 2)
        self.conv2 = nn.Conv2d(planes * 2, planes * 4, kernel_size=3,
                               stride=stride, padding=1, groups=groups,
                               bias=False)
        self.bn2 = nn.BatchNorm2d(planes * 4)
        self.conv3 = nn.Conv2d(planes * 4, planes * 4, kernel_size=1,
                               bias=False)
        self.bn3 = nn.BatchNorm2d(planes * 4)
        self.relu = nn.ReLU(inplace=True)
        self.se_module = SEModule(planes * 4, reduction=reduction)
        self.downsample = downsample
        self.stride = stride


class SEResNetBottleneck(Bottleneck):
    """
    ResNet bottleneck with a Squeeze-and-Excitation module. It follows Caffe
    implementation and uses `stride=stride` in `conv1` and not in `conv2`
    (the latter is used in the torchvision implementation of ResNet).
    """
    expansion = 4

    def __init__(self, inplanes, planes, groups, reduction, stride=1,
                 downsample=None):
        super(SEResNetBottleneck, self).__init__()
        self.conv1 = nn.Conv2d(inplanes, planes, kernel_size=1, bias=False,
                               stride=stride)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, padding=1,
                               groups=groups, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv3 = nn.Conv2d(planes, planes * 4, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(planes * 4)
        self.relu = nn.ReLU(inplace=True)
        self.se_module = SEModule(planes * 4, reduction=reduction)
        self.downsample = downsample
        self.stride = stride


class SEResNeXtBottleneck(Bottleneck):
    """
    ResNeXt bottleneck type C with a Squeeze-and-Excitation module.
    """
    expansion = 4

    def __init__(self, inplanes, planes, groups, reduction, stride=1,
                 downsample=None, base_width=4):
        super(SEResNeXtBottleneck, self).__init__()
        width = math.floor(planes * (base_width / 64)) * groups
        self.conv1 = nn.Conv2d(inplanes, width, kernel_size=1, bias=False,
                               stride=1)
        self.bn1 = nn.BatchNorm2d(width)
        self.conv2 = nn.Conv2d(width, width, kernel_size=3, stride=stride,
                               padding=1, groups=groups, bias=False)
        self.bn2 = nn.BatchNorm2d(width)
        self.conv3 = nn.Conv2d(width, planes * 4, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(planes * 4)
        self.relu = nn.ReLU(inplace=True)
        self.se_module = SEModule(planes * 4, reduction=reduction)
        self.downsample = downsample
        self.stride = stride


class SENet(nn.Module):

    def __init__(self, block, layers, groups, reduction, dropout_p=0.2,
                 inplanes=128, input_3x3=True, downsample_kernel_size=3,
                 downsample_padding=1, num_classes=1000):
        """
        Parameters
        ----------
        block (nn.Module): Bottleneck class.
            - For SENet154: SEBottleneck
            - For SE-ResNet models: SEResNetBottleneck
            - For SE-ResNeXt models:  SEResNeXtBottleneck
        layers (list of ints): Number of residual blocks for 4 layers of the
            network (layer1...layer4).
        groups (int): Number of groups for the 3x3 convolution in each
            bottleneck block.
            - For SENet154: 64
            - For SE-ResNet models: 1
            - For SE-ResNeXt models:  32
        reduction (int): Reduction ratio for Squeeze-and-Excitation modules.
            - For all models: 16
        dropout_p (float or None): Drop probability for the Dropout layer.
            If `None` the Dropout layer is not used.
            - For SENet154: 0.2
            - For SE-ResNet models: None
            - For SE-ResNeXt models: None
        inplanes (int):  Number of input channels for layer1.
            - For SENet154: 128
            - For SE-ResNet models: 64
            - For SE-ResNeXt models: 64
        input_3x3 (bool): If `True`, use three 3x3 convolutions instead of
            a single 7x7 convolution in layer0.
            - For SENet154: True
            - For SE-ResNet models: False
            - For SE-ResNeXt models: False
        downsample_kernel_size (int): Kernel size for downsampling convolutions
            in layer2, layer3 and layer4.
            - For SENet154: 3
            - For SE-ResNet models: 1
            - For SE-ResNeXt models: 1
        downsample_padding (int): Padding for downsampling convolutions in
            layer2, layer3 and layer4.
            - For SENet154: 1
            - For SE-ResNet models: 0
            - For SE-ResNeXt models: 0
        num_classes (int): Number of outputs in `last_linear` layer.
            - For all models: 1000
        """
        super(SENet, self).__init__()
        self.inplanes = inplanes
        if input_3x3:
            layer0_modules = [
                ('conv1', nn.Conv2d(3, 64, 3, stride=2, padding=1,
                                    bias=False)),
                ('bn1', nn.BatchNorm2d(64)),
                ('relu1', nn.ReLU(inplace=True)),
                ('conv2', nn.Conv2d(64, 64, 3, stride=1, padding=1,
                                    bias=False)),
                ('bn2', nn.BatchNorm2d(64)),
                ('relu2', nn.ReLU(inplace=True)),
                ('conv3', nn.Conv2d(64, inplanes, 3, stride=1, padding=1,
                                    bias=False)),
                ('bn3', nn.BatchNorm2d(inplanes)),
                ('relu3', nn.ReLU(inplace=True)),
            ]
        else:
            layer0_modules = [
                ('conv1', nn.Conv2d(3, inplanes, kernel_size=7, stride=2,
                                    padding=3, bias=False)),
                ('bn1', nn.BatchNorm2d(inplanes)),
                ('relu1', nn.ReLU(inplace=True)),
            ]
        # To preserve compatibility with Caffe weights `ceil_mode=True`
        # is used instead of `padding=1`.
        layer0_modules.append(('pool', nn.MaxPool2d(3, stride=2,
                                                    ceil_mode=True)))
        self.layer0 = nn.Sequential(OrderedDict(layer0_modules))
        self.layer1 = self._make_layer(
            block,
            planes=64,
            blocks=layers[0],
            groups=groups,
            reduction=reduction,
            downsample_kernel_size=1,
            downsample_padding=0
        )
        self.layer2 = self._make_layer(
            block,
            planes=128,
            blocks=layers[1],
            stride=2,
            groups=groups,
            reduction=reduction,
            downsample_kernel_size=downsample_kernel_size,
            downsample_padding=downsample_padding
        )
        self.layer3 = self._make_layer(
            block,
            planes=256,
            blocks=layers[2],
            stride=2,
            groups=groups,
            reduction=reduction,
            downsample_kernel_size=downsample_kernel_size,
            downsample_padding=downsample_padding
        )
        self.layer4 = self._make_layer(
            block,
            planes=512,
            blocks=layers[3],
            stride=2,
            groups=groups,
            reduction=reduction,
            downsample_kernel_size=downsample_kernel_size,
            downsample_padding=downsample_padding
        )
        self.avg_pool = nn.AvgPool2d(7, stride=1)
        self.dropout = nn.Dropout(dropout_p) if dropout_p is not None else None
        self.last_linear = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block, planes, blocks, groups, reduction, stride=1,
                    downsample_kernel_size=1, downsample_padding=0):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.inplanes, planes * block.expansion,
                          kernel_size=downsample_kernel_size, stride=stride,
                          padding=downsample_padding, bias=False),
                nn.BatchNorm2d(planes * block.expansion),
            )

        layers = []
        layers.append(block(self.inplanes, planes, groups, reduction, stride,
                            downsample))
        self.inplanes = planes * block.expansion
        for i in range(1, blocks):
            layers.append(block(self.inplanes, planes, groups, reduction))

        return nn.Sequential(*layers)

    def features(self, x):
        x = self.layer0(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return x

    def logits(self, x):
        x = self.avg_pool(x)
        if self.dropout is not None:
            x = self.dropout(x)
        x = x.view(x.size(0), -1)
        x = self.last_linear(x)
        return x

    def forward(self, x):
        x = self.features(x)
        x = self.logits(x)
        return x


def initialize_pretrained_model(model, num_classes, settings):
    assert num_classes == settings['num_classes'], \
        'num_classes should be {}, but is {}'.format(
            settings['num_classes'], num_classes)
    model.load_state_dict(torch.load(settings['file_path']))
    model.input_space = settings['input_space']
    model.input_size = settings['input_size']
    model.input_range = settings['input_range']
    model.mean = settings['mean']
    model.std = settings['std']

def se_resnext50_32x4d(num_classes=1000, pretrained='imagenet'):
    model = SENet(SEResNeXtBottleneck, [3, 4, 6, 3], groups=32, reduction=16,
                  dropout_p=None, inplanes=64, input_3x3=False,
                  downsample_kernel_size=1, downsample_padding=0,
                  num_classes=num_classes)
    if pretrained is not None:
        settings = pretrained_settings['se_resnext50_32x4d'][pretrained]
        initialize_pretrained_model(model, num_classes, settings)
    return model


# 4. Training Utitility Functions

In [11]:
################
# Optimizer
################

class RAdam(torch.optim.Optimizer):

    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0, degenerated_to_sgd=True):
        if not 0.0 <= lr:
            raise ValueError("Invalid learning rate: {}".format(lr))
        if not 0.0 <= eps:
            raise ValueError("Invalid epsilon value: {}".format(eps))
        if not 0.0 <= betas[0] < 1.0:
            raise ValueError("Invalid beta parameter at index 0: {}".format(betas[0]))
        if not 0.0 <= betas[1] < 1.0:
            raise ValueError("Invalid beta parameter at index 1: {}".format(betas[1]))
        
        self.degenerated_to_sgd = degenerated_to_sgd
        if isinstance(params, (list, tuple)) and len(params) > 0 and isinstance(params[0], dict):
            for param in params:
                if 'betas' in param and (param['betas'][0] != betas[0] or param['betas'][1] != betas[1]):
                    param['buffer'] = [[None, None, None] for _ in range(10)]
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay, buffer=[[None, None, None] for _ in range(10)])
        super(RAdam, self).__init__(params, defaults)

    def __setstate__(self, state):
        super(RAdam, self).__setstate__(state)

    def step(self, closure=None):

        loss = None
        if closure is not None:
            loss = closure()

        for group in self.param_groups:

            for p in group['params']:
                if p.grad is None:
                    continue
                grad = p.grad.data.float()
                if grad.is_sparse:
                    raise RuntimeError('RAdam does not support sparse gradients')

                p_data_fp32 = p.data.float()

                state = self.state[p]

                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p_data_fp32)
                    state['exp_avg_sq'] = torch.zeros_like(p_data_fp32)
                else:
                    state['exp_avg'] = state['exp_avg'].type_as(p_data_fp32)
                    state['exp_avg_sq'] = state['exp_avg_sq'].type_as(p_data_fp32)

                exp_avg, exp_avg_sq = state['exp_avg'], state['exp_avg_sq']
                beta1, beta2 = group['betas']

                exp_avg_sq.mul_(beta2).addcmul_(1 - beta2, grad, grad)
                exp_avg.mul_(beta1).add_(1 - beta1, grad)

                state['step'] += 1
                buffered = group['buffer'][int(state['step'] % 10)]
                if state['step'] == buffered[0]:
                    N_sma, step_size = buffered[1], buffered[2]
                else:
                    buffered[0] = state['step']
                    beta2_t = beta2 ** state['step']
                    N_sma_max = 2 / (1 - beta2) - 1
                    N_sma = N_sma_max - 2 * state['step'] * beta2_t / (1 - beta2_t)
                    buffered[1] = N_sma

                    # more conservative since it's an approximated value
                    if N_sma >= 5:
                        step_size = math.sqrt((1 - beta2_t) * (N_sma - 4) / (N_sma_max - 4) * (N_sma - 2) / N_sma * N_sma_max / (N_sma_max - 2)) / (1 - beta1 ** state['step'])
                    elif self.degenerated_to_sgd:
                        step_size = 1.0 / (1 - beta1 ** state['step'])
                    else:
                        step_size = -1
                    buffered[2] = step_size

                # more conservative since it's an approximated value
                if N_sma >= 5:
                    if group['weight_decay'] != 0:
                        p_data_fp32.add_(-group['weight_decay'] * group['lr'], p_data_fp32)
                    denom = exp_avg_sq.sqrt().add_(group['eps'])
                    p_data_fp32.addcdiv_(-step_size * group['lr'], exp_avg, denom)
                    p.data.copy_(p_data_fp32)
                elif step_size > 0:
                    if group['weight_decay'] != 0:
                        p_data_fp32.add_(-group['weight_decay'] * group['lr'], p_data_fp32)
                    p_data_fp32.add_(-step_size * group['lr'], exp_avg)
                    p.data.copy_(p_data_fp32)

        return loss

In [12]:
################
# Custom Loss Function
################

def loss_function(y_preds, y_true):
    
    logit1, logit2, logit3, logit4 = y_preds[:,: 168], y_preds[:,168: 168+11], y_preds[:,168+11:168+11+7], y_preds[:,168+11+7:]
    label1, label2, label3, label4 = y_true[:,0], y_true[:,1], y_true[:,2], y_true[:,3]
    
    grapheme_root_loss = nn.CrossEntropyLoss()(logit1, label1) # BCEWithLogitsLoss
    vowel_diacritic_loss = nn.CrossEntropyLoss()(logit2, label2)
    consonant_diacritic_loss = nn.CrossEntropyLoss()(logit3, label3)
    grapheme_id_loss = nn.CrossEntropyLoss()(logit4, label4)
    
    return 0.2 * grapheme_root_loss + 0.1 * vowel_diacritic_loss + \
            0.1 * consonant_diacritic_loss + 0.4 * grapheme_id_loss


In [13]:
################
# 4 logit outputs to predicted 3 labels
################

grapheme_root_matrix = np.zeros((168, 1295))
vowel_diacritic_matrix = np.zeros((11, 1295))
consonant_diacritic_matrix = np.zeros((7, 1295))
for i, value in grapheme_dict.items():
    grapheme_root_matrix[value['grapheme_root']][i] = 1.
    vowel_diacritic_matrix[value['vowel_diacritic']][i] = 1.
    consonant_diacritic_matrix[value['consonant_diacritic']][i] = 1.

def inference_from_raw_logits(y_preds, grapheme_dict=grapheme_dict, grapheme_root_matrix=grapheme_root_matrix, vowel_diacritic_matrix=vowel_diacritic_matrix, consonant_diacritic_matrix=consonant_diacritic_matrix, raw_output=False):
    
    logit1, logit2, logit3, logit4 = y_preds[:,: 168], y_preds[:,168: 168+11], y_preds[:,168+11:168+11+7], y_preds[:,168+11+7:]
    threelogits = np.matmul(logit1, grapheme_root_matrix) / 3 + np.matmul(logit2, vowel_diacritic_matrix) / 3 + np.matmul(logit3, consonant_diacritic_matrix) / 3
    if raw_output:
        return threelogits / 2 + logit4 / 2
    grapheme_id_preds = np.argmax(threelogits / 2 + logit4 / 2, axis=1)
    
    f = lambda x: grapheme_dict[x]['grapheme_root']
    f = np.vectorize(f)
    grapheme_root = f(grapheme_id_preds)
    f = lambda x: grapheme_dict[x]['vowel_diacritic']
    f = np.vectorize(f)
    vowel_diacritic = f(grapheme_id_preds)
    f = lambda x: grapheme_dict[x]['consonant_diacritic']
    f = np.vectorize(f)
    consonant_diacritic = f(grapheme_id_preds)
    
    return grapheme_root, vowel_diacritic, consonant_diacritic


In [14]:
################
# Cutmix
################

def rand_bbox(size, lam):
    W = size[2]
    H = size[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w = np.int(W * cut_rat)
    cut_h = np.int(H * cut_rat)

    # uniform
    cx = np.random.randint(W)
    cy = np.random.randint(H)

    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)

    return bbx1, bby1, bbx2, bby2

def cutmix(data, labels, alpha=1.0):
    indices = torch.randperm(data.size(0))
    labels1 = labels[:,0]
    labels2 = labels[:,1]
    labels3 = labels[:,2]
    labels4 = labels[:,3]
    shuffled_labels1 = labels1[indices]
    shuffled_labels2 = labels2[indices]
    shuffled_labels3 = labels3[indices]
    shuffled_labels4 = labels4[indices]

    lam = np.random.beta(alpha, alpha)
    bbx1, bby1, bbx2, bby2 = rand_bbox(data.size(), lam)
    data[:, :, bbx1:bbx2, bby1:bby2] = data[indices, :, bbx1:bbx2, bby1:bby2]
    # adjust lambda to exactly match pixel ratio
    lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (data.size()[-1] * data.size()[-2]))

    labels = [labels1, shuffled_labels1, labels2, shuffled_labels2, labels3, shuffled_labels3, labels4, shuffled_labels4, lam]
    return data, labels

def cutmix_loss_function(y_preds, y_true):
    logit1, logit2, logit3, logit4 = y_preds[:,: 168], y_preds[:,168: 168+11], y_preds[:,168+11:168+11+7], y_preds[:,168+11+7:]
    labels1, shuffled_labels1, labels2, shuffled_labels2, labels3, shuffled_labels3, labels4, shuffled_labels4, lam = y_true
    c = nn.CrossEntropyLoss(reduction='mean')
    return 0.2 * (lam * c(logit1, labels1) + (1 - lam) * c(logit1, shuffled_labels1)) + \
            0.1 * (lam * c(logit2, labels2) + (1 - lam) * c(logit2, shuffled_labels2)) + \
            0.1 * (lam * c(logit3, labels3) + (1 - lam) * c(logit3, shuffled_labels3)) + \
            0.4 * (lam * c(logit4, labels4) + (1 - lam) * c(logit4, shuffled_labels4))


In [15]:
################
# Mixup
################

def mixup(data, labels, alpha=1.0):
    indices = torch.randperm(data.size(0))
    shuffled_data = data[indices]
    labels1 = labels[:,0]
    labels2 = labels[:,1]
    labels3 = labels[:,2]
    labels4 = labels[:,3]
    shuffled_labels1 = labels1[indices]
    shuffled_labels2 = labels2[indices]
    shuffled_labels3 = labels3[indices]
    shuffled_labels4 = labels4[indices]

    lam = np.random.beta(alpha, alpha)
    data = data * lam + shuffled_data * (1 - lam)
    
    labels = [labels1, shuffled_labels1, labels2, shuffled_labels2, labels3, shuffled_labels3, labels4, shuffled_labels4, lam]
    return data, labels

def mixup_loss_function(y_preds, y_true):
    logit1, logit2, logit3, logit4 = y_preds[:,: 168], y_preds[:,168: 168+11], y_preds[:,168+11:168+11+7], y_preds[:,168+11+7:]
    labels1, shuffled_labels1, labels2, shuffled_labels2, labels3, shuffled_labels3, labels4, shuffled_labels4, lam = y_true
    c = nn.CrossEntropyLoss(reduction='mean')
    return 0.2 * (lam * c(logit1, labels1) + (1 - lam) * c(logit1, shuffled_labels1)) + \
            0.1 * (lam * c(logit2, labels2) + (1 - lam) * c(logit2, shuffled_labels2)) + \
            0.1 * (lam * c(logit3, labels3) + (1 - lam) * c(logit3, shuffled_labels3)) + \
            0.4 * (lam * c(logit4, labels4) + (1 - lam) * c(logit4, shuffled_labels4))


In [16]:
################
# FMix
################

def fftfreqnd(h, w=None, z=None):
    """ Get bin values for discrete fourier transform of size (h, w, z)
    :param h: Required, first dimension size
    :param w: Optional, second dimension size
    :param z: Optional, third dimension size
    """
    fz = fx = 0
    fy = np.fft.fftfreq(h)

    if w is not None:
        fy = np.expand_dims(fy, -1)

        if w % 2 == 1:
            fx = np.fft.fftfreq(w)[: w // 2 + 2]
        else:
            fx = np.fft.fftfreq(w)[: w // 2 + 1]

    if z is not None:
        fy = np.expand_dims(fy, -1)
        if z % 2 == 1:
            fz = np.fft.fftfreq(z)[:, None]
        else:
            fz = np.fft.fftfreq(z)[:, None]

    return np.sqrt(fx * fx + fy * fy + fz * fz)


def get_spectrum(freqs, decay_power, ch, h, w=0, z=0):
    """ Samples a fourier image with given size and frequencies decayed by decay power
    :param freqs: Bin values for the discrete fourier transform
    :param decay_power: Decay power for frequency decay prop 1/f**d
    :param ch: Number of channels for the resulting mask
    :param h: Required, first dimension size
    :param w: Optional, second dimension size
    :param z: Optional, third dimension size
    """
    scale = np.ones(1) / (np.maximum(freqs, np.array([1. / max(w, h, z)])) ** decay_power)

    param_size = [ch] + list(freqs.shape) + [2]
    param = np.random.randn(*param_size)

    scale = np.expand_dims(scale, -1)[None, :]

    return scale * param


def make_low_freq_image(decay, shape, ch=1):
    """ Sample a low frequency image from fourier space
    :param decay_power: Decay power for frequency decay prop 1/f**d
    :param shape: Shape of desired mask, list up to 3 dims
    :param ch: Number of channels for desired mask
    """
    freqs = fftfreqnd(*shape)
    spectrum = get_spectrum(freqs, decay, ch, *shape)#.reshape((1, *shape[:-1], -1))
    spectrum = spectrum[:, 0] + 1j * spectrum[:, 1]
    mask = np.real(np.fft.irfftn(spectrum, shape))

    if len(shape) == 1:
        mask = mask[:1, :shape[0]]
    if len(shape) == 2:
        mask = mask[:1, :shape[0], :shape[1]]
    if len(shape) == 3:
        mask = mask[:1, :shape[0], :shape[1], :shape[2]]

    mask = mask
    mask = (mask - mask.min())
    mask = mask / mask.max()
    return mask


def sample_lam(alpha, reformulate=False):
    """ Sample a lambda from symmetric beta distribution with given alpha
    :param alpha: Alpha value for beta distribution
    :param reformulate: If True, uses the reformulation of [1].
    """
    if reformulate:
        lam = scs.beta.rvs(alpha+1, alpha)
    else:
        lam = scs.beta.rvs(alpha, alpha)

    return lam


def binarise_mask(mask, lam, in_shape, max_soft=0.0):
    """ Binarises a given low frequency image such that it has mean lambda.
    :param mask: Low frequency image, usually the result of `make_low_freq_image`
    :param lam: Mean value of final mask
    :param in_shape: Shape of inputs
    :param max_soft: Softening value between 0 and 0.5 which smooths hard edges in the mask.
    :return:
    """
    idx = mask.reshape(-1).argsort()[::-1]
    mask = mask.reshape(-1)
    num = math.ceil(lam * mask.size) if random.random() > 0.5 else math.floor(lam * mask.size)

    eff_soft = max_soft
    if max_soft > lam or max_soft > (1-lam):
        eff_soft = min(lam, 1-lam)

    soft = int(mask.size * eff_soft)
    num_low = num - soft
    num_high = num + soft

    mask[idx[:num_high]] = 1
    mask[idx[num_low:]] = 0
    mask[idx[num_low:num_high]] = np.linspace(1, 0, (num_high - num_low))

    mask = mask.reshape((1, *in_shape))
    return mask


def sample_mask(alpha, decay_power, shape, max_soft=0.0, reformulate=False):
    """ Samples a mean lambda from beta distribution parametrised by alpha, creates a low frequency image and binarises
    it based on this lambda
    :param alpha: Alpha value for beta distribution from which to sample mean of mask
    :param decay_power: Decay power for frequency decay prop 1/f**d
    :param shape: Shape of desired mask, list up to 3 dims
    :param max_soft: Softening value between 0 and 0.5 which smooths hard edges in the mask.
    :param reformulate: If True, uses the reformulation of [1].
    """
    if isinstance(shape, int):
        shape = (shape,)

    # Choose lambda
    lam = sample_lam(alpha, reformulate)

    # Make mask, get mean / std
    mask = make_low_freq_image(decay_power, shape)
    mask = binarise_mask(mask, lam, shape, max_soft)

    return lam, mask


def sample_and_apply(x, alpha, decay_power, shape, max_soft=0.0, reformulate=False):
    """
    :param x: Image batch on which to apply fmix of shape [b, c, shape*]
    :param alpha: Alpha value for beta distribution from which to sample mean of mask
    :param decay_power: Decay power for frequency decay prop 1/f**d
    :param shape: Shape of desired mask, list up to 3 dims
    :param max_soft: Softening value between 0 and 0.5 which smooths hard edges in the mask.
    :param reformulate: If True, uses the reformulation of [1].
    :return: mixed input, permutation indices, lambda value of mix,
    """
    lam, mask = sample_mask(alpha, decay_power, shape, max_soft, reformulate)
    index = np.random.permutation(x.shape[0])

    x1, x2 = x * torch.from_numpy(mask).to(DEVICE), x[index] * torch.from_numpy(1-mask).to(DEVICE)
    return (x1+x2).float(), index, lam

def fmix(data, labels, alpha=1.0):
    
    data, indices, lam = sample_and_apply(data, alpha=alpha, decay_power=3, shape=(IMAGE_SIZE, IMAGE_SIZE), max_soft=0.1, reformulate=False)
    
    labels1 = labels[:,0]
    labels2 = labels[:,1]
    labels3 = labels[:,2]
    labels4 = labels[:,3]
    shuffled_labels1 = labels1[indices]
    shuffled_labels2 = labels2[indices]
    shuffled_labels3 = labels3[indices]
    shuffled_labels4 = labels4[indices]

    labels = [labels1, shuffled_labels1, labels2, shuffled_labels2, labels3, shuffled_labels3, labels4, shuffled_labels4, lam]
    return data, labels

def fmix_loss_function(y_preds, y_true):
    logit1, logit2, logit3, logit4 = y_preds[:,: 168], y_preds[:,168: 168+11], y_preds[:,168+11:168+11+7], y_preds[:,168+11+7:]
    labels1, shuffled_labels1, labels2, shuffled_labels2, labels3, shuffled_labels3, labels4, shuffled_labels4, lam = y_true
    c = nn.CrossEntropyLoss(reduction='mean')
    return 0.2 * (lam * c(logit1, labels1) + (1 - lam) * c(logit1, shuffled_labels1)) + \
            0.1 * (lam * c(logit2, labels2) + (1 - lam) * c(logit2, shuffled_labels2)) + \
            0.1 * (lam * c(logit3, labels3) + (1 - lam) * c(logit3, shuffled_labels3)) + \
            0.4 * (lam * c(logit4, labels4) + (1 - lam) * c(logit4, shuffled_labels4))


In [17]:
################
# Training Validating and Predicting Functions
################

def training_with_accumulation(model, train_loader, optimizer, criterion, scheduler, apply_cutmix=False, apply_mixup=False, apply_fmix=False):
    
    model.train()
    avg_loss = 0.
    optimizer.zero_grad()

    bar = tqdm.tqdm(
        enumerate(train_loader), 
        total=len(train_loader), 
        postfix={"train_loss":0.0,}
    )
    for idx, batch in bar:
        
        random_float = np.random.rand()
        
        data, labels = batch
        data, labels = data.to(DEVICE), labels.to(DEVICE)
        if apply_cutmix and (random_float<=0.33):
            data, labels = cutmix(data, labels)
        elif apply_mixup and (random_float>0.33) and (random_float<=0.66):
            data, labels = mixup(data, labels)
        elif apply_fmix and (random_float>1.) and (random_float<=1.):
            data, labels = fmix(data, labels)
        
        logits = model(data)
        if apply_cutmix and (random_float<=0.33):
            loss = cutmix_loss_function(logits, labels)
        elif apply_mixup and (random_float>0.33) and (random_float<=0.66):
            loss = mixup_loss_function(logits, labels)
        elif apply_fmix and (random_float>1.) and (random_float<=1.):
            loss = fmix_loss_function(logits, labels)
        else:
            loss = criterion(logits, labels)
        loss.backward()
        if (idx + 1) % BATCH_ACCUMULATION_COUNT == 0:    
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        
        avg_loss += loss.item() / (len(train_loader))
        
        bar.set_postfix(ordered_dict={
            "train_loss":loss.item(),
        })
        del data, labels

    torch.cuda.empty_cache()
    gc.collect()
    
    return avg_loss


def validate_model(model, criterion, val_loader, target_cols=TARGET_COLS, batch_size=VALIDATION_BATCH_SIZE, verbose=False):

    avg_val_loss = 0.
    model.eval()
    
    y_preds = np.zeros((len(val_loader.dataset), 168+11+7+1295))
    y_true = np.zeros((len(val_loader.dataset), 4))
    
    with torch.no_grad():
        
        for idx, batch in enumerate(val_loader):
            data, labels = batch
            data, labels = data.to(DEVICE), labels.to(DEVICE)
            
            logits = model(data)
            logits = torch.sigmoid(logits)
            
            avg_val_loss += criterion(logits, labels).item() / len(val_loader)
            logits = logits.detach().cpu().squeeze().numpy()
            labels = labels.detach().cpu().squeeze().numpy()
            y_preds[idx*batch_size : (idx+1)*batch_size] = logits
            y_true[idx*batch_size : (idx+1)*batch_size]  = labels
            
            del logits, labels
            
        torch.cuda.empty_cache()
        gc.collect()
        
        grapheme_root, vowel_diacritic, consonant_diacritic = inference_from_raw_logits(y_preds)
        label1, label2, label3, _ = y_true.T
        
        score = 0.5 * recall_score(grapheme_root, label1, average='macro') + 0.25 * recall_score(vowel_diacritic, label2, average='macro') + 0.25 * recall_score(consonant_diacritic, label3, average='macro')
        if verbose:
            print('grapheme_root score : ', recall_score(grapheme_root, label1, average='macro'))
            print('vowel_diacritic score : ', recall_score(vowel_diacritic, label2, average='macro'))
            print('consonant_diacritic score : ', recall_score(consonant_diacritic, label3, average='macro'))
            print('RAW OUTPUT grapheme_root score : ', recall_score(np.argmax(y_preds[:,: 168], axis=1), y_true.T[0], average='macro'))
            print('RAW OUTPUT vowel_diacritic score : ', recall_score(np.argmax(y_preds[:,168: 168+11], axis=1), y_true.T[1], average='macro'))
            print('RAW OUTPUT consonant_diacritic score : ', recall_score(np.argmax(y_preds[:,168+11:168+11+7], axis=1), y_true.T[2], average='macro'))
            print('RAW OUTPUT grapheme_id score : ', recall_score(np.argmax(y_preds[:,168+11+7:], axis=1), y_true.T[3], average='macro'))
        
    return avg_val_loss, score


def predict(model, test_loader, target_cols=TARGET_COLS, batch_size=VALIDATION_BATCH_SIZE, raw_output=False):
    
    test_preds = np.zeros((len(test_loader.dataset), 168+11+7+1295))
    
    model.eval()
    with torch.no_grad():
        for idx, x_batch in tqdm.tqdm(enumerate(test_loader)):
            data = x_batch
            data = data.to(DEVICE)
            predictions = model(data)
            predictions = torch.sigmoid(predictions)
            test_preds[idx*batch_size : (idx+1)*batch_size] = predictions.detach().cpu().squeeze().numpy()

    if raw_output:
        return inference_from_raw_logits(test_preds, raw_output=raw_output)
    
    grapheme_root, vowel_diacritic, consonant_diacritic = inference_from_raw_logits(test_preds, raw_output=raw_output)
    return grapheme_root, vowel_diacritic, consonant_diacritic


# 5. Training Starts Here

In [18]:
################
# Preprocessing
################

if TRAINING:
    train_images = prepare_images(data_type='train')
    train_labels = train[['grapheme_root','vowel_diacritic','consonant_diacritic','grapheme_id']].values
    

/home/ec2-user/anaconda3/envs/pytorch_p36/lib/python3.6/site-packages/pandas/io/feather_format.py:124: FutureWarning: `nthreads` argument is deprecated, pass `use_threads` instead
  nthreads=int_use_threads)
/home/ec2-user/anaconda3/envs/pytorch_p36/lib/python3.6/site-packages/pyarrow/pandas_compat.py:751: FutureWarning: .labels was deprecated in version 0.24.0. Use .codes instead.
  labels, = index.labels
50210it [00:58, 863.76it/s]
50210it [00:58, 864.71it/s]
50210it [00:58, 861.09it/s]
50210it [01:00, 826.36it/s]


In [ ]:
################
# Training Part
################

if TRAINING:
    
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    for fold, (trn_idx, val_idx) in enumerate(skf.split(X=train.image_id, y=train.grapheme_id)):
        
        if fold not in FOLD:
            continue
            
        train_transform = Transform(elastic_distortion_ratio=0.2, grid_mask_ratio=0.5)
        valid_transform = Transform(affine=False)

        train_dataset = BengaliAIDataset(train_images, train_labels, transform=train_transform, indices=trn_idx)
        valid_dataset = BengaliAIDataset(train_images, train_labels, transform=valid_transform, indices=val_idx)
        
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=True)
        valid_loader = DataLoader(valid_dataset, batch_size=VALIDATION_BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=False)
        
        model = se_resnext50_32x4d(num_classes=1000, pretrained='imagenet')
        if EPOCH_RELEASE > 0:
            for param in model.parameters():
                param.requires_grad = False
        model.layer0.conv1 = nn.Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
        model.avg_pool = GeM()
        model.last_linear = nn.Linear(model.last_linear.in_features, 168+11+7+1295)
        model.to(DEVICE)
        model.zero_grad()
        torch.cuda.empty_cache()
        
        model.train()
        
        criterion = loss_function
        optimizer = RAdam(model.parameters(), lr=1e-3)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20, eta_min=2e-6)
        
        train_start_time = datetime.datetime.now()
        best_score = 0.0
        for epoch in range(EPOCHS):
            epoch_start_time = datetime.datetime.now()
            torch.cuda.empty_cache()

            if epoch == EPOCH_RELEASE:
                for param in model.parameters():
                    param.requires_grad = True
                    
            avg_loss = training_with_accumulation(
                model, train_loader, optimizer, criterion, scheduler, apply_cutmix=True, apply_mixup=True, apply_fmix=False)
            avg_val_loss, val_score = validate_model(model, criterion, valid_loader)

            print("Epoch {} : {} seconds : train loss {:.4f} : valid loss {:.4f} : valid score {:.4f}".format(
                epoch, (datetime.datetime.now() - epoch_start_time).seconds, avg_loss, avg_val_loss, val_score))

            if val_score > best_score:
                best_score = val_score
                torch.save(model.state_dict(), os.path.join("./model_{}_{}.ckpt".format(VERSION, fold)))
                early_stopping_count = 0
            else:
                early_stopping_count += 1
                if early_stopping_count == EARLY_STOPPING:
                    print("Early Stopping : ", epoch)
                    break

        print('-'*20)
        print("Fold {} : Total Training Time {}, Best Score : {}".format(
            fold, datetime.datetime.now()-train_start_time, best_score))
        print('-'*20)
        
        model.load_state_dict(torch.load(os.path.join("./model_{}_{}.ckpt".format(VERSION, fold))))
        avg_val_loss, val_spearmanr = validate_model(
            model, criterion, valid_loader, verbose=True)
        best_scores.append(val_score)
        
        del model
        gc.collect()

100%|██████████| 2511/2511 [10:15<00:00,  4.08it/s, train_loss=3.81]


Epoch 0 : 679 seconds : train loss 4.1477 : valid loss 4.0934 : valid score 0.2655


/home/ec2-user/anaconda3/envs/pytorch_p36/lib/python3.6/site-packages/sklearn/metrics/classification.py:1145: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples.
  'recall', 'true', average, warn_for)
100%|██████████| 2511/2511 [13:27<00:00,  3.11it/s, train_loss=2.74] 


Epoch 1 : 864 seconds : train loss 2.4663 : valid loss 3.7745 : valid score 0.9223


100%|██████████| 2511/2511 [13:27<00:00,  3.11it/s, train_loss=1.87] 


...

100%|██████████| 2511/2511 [13:27<00:00,  3.11it/s, train_loss=1.23]    


Epoch 136 : 865 seconds : train loss 0.6142 : valid loss 3.6284 : valid score 0.9889


100%|██████████| 2511/2511 [13:27<00:00,  3.11it/s, train_loss=0.0203] 


Epoch 137 : 865 seconds : train loss 0.6466 : valid loss 3.6292 : valid score 0.9875


 29%|██▉       | 732/2511 [03:56<09:35,  3.09it/s, train_loss=0.0386] 

In [20]:
if VALIDATION:
    model.load_state_dict(torch.load(os.path.join("./model_{}_{}.ckpt".format(VERSION, fold))))
    avg_val_loss, val_score = validate_model(model, criterion, valid_loader, verbose=True)
    print("Overall Score : ", val_score)

grapheme_root score :  0.9867487182199294
vowel_diacritic score :  0.9953658853474234
consonant_diacritic score :  0.9948117783072556
RAW OUTPUT grapheme_root score :  0.985145663389779
RAW OUTPUT vowel_diacritic score :  0.993431205283606
RAW OUTPUT consonant_diacritic score :  0.9934339758719265
RAW OUTPUT grapheme_id score :  0.9844807003552506
Overall Score :  0.9909187750236343
